# 💻 Unidad 1: Material Complementario - Práctica
## Módulo 02 - Manejo de Datos Faltantes y Outliers
### Laboratorio (Herramientas) - Universidad del Aconcagua

---

## 🎯 Objetivos

1. ✅ Analizar patrones de datos faltantes en dataset real
2. ✅ Aplicar múltiples estrategias de imputación y comparar resultados
3. ✅ Detectar outliers con IQR, Z-score e Isolation Forest
4. ✅ Decidir tratamiento apropiado según contexto
5. ✅ Validar impacto en modelos ML

### 📋 Ejercicios

1. **Ejercicio 1**: Diagnóstico de Datos Faltantes
2. **Ejercicio 2**: Imputación y Comparación de Métodos
3. **Ejercicio 3**: Detección de Outliers (múltiples métodos)
4. **Ejercicio 4**: Tratamiento de Outliers
5. **Ejercicio 5**: Validación en Modelo ML

### ⏱️ Duración: 60 minutos

In [0]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import IsolationForest
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

---

## 📊 Ejercicio 1: Diagnóstico de Datos Faltantes

**Objetivo**: Analizar patrones de valores faltantes en el dataset de ventas

In [0]:
# Cargar y simular datos faltantes
ventas = pd.read_csv("/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/ventas.csv")

# Simular missing de forma realista (MAR)
np.random.seed(42)
ventas_missing = ventas.copy().head(1000)

# Simular missing en total para ventas de fin de semana (MAR)
ventas_missing.loc[ventas_missing['es_fin_de_semana'], 'total'] = np.where(
    np.random.rand(len(ventas_missing[ventas_missing['es_fin_de_semana']])) < 0.3,
    np.nan,
    ventas_missing.loc[ventas_missing['es_fin_de_semana'], 'total']
)

# Missing aleatorio en sucursal_id (MCAR)
ventas_missing.loc[np.random.choice(ventas_missing.index, 50, replace=False), 'sucursal_id'] = np.nan

print(f"📊 Dataset con missing simulado:")
print(f"  Filas: {len(ventas_missing)}")
print(f"\n🚨 Valores faltantes por columna:")
missing_counts = ventas_missing.isnull().sum()
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

In [0]:
# Visualizar patrones de missing
plt.figure(figsize=(12, 4))

# Heatmap de missing
plt.subplot(1, 2, 1)
sns.heatmap(ventas_missing.isnull(), cbar=False, yticklabels=False)
plt.title("Patrón de Valores Faltantes")

# Porcentaje por columna
plt.subplot(1, 2, 2)
missing_pct = (ventas_missing.isnull().sum() / len(ventas_missing) * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0].plot(kind='barh')
plt.xlabel("Porcentaje Missing (%)")
plt.title("Missing por Columna")

plt.tight_layout()
plt.show()

print(f"\n💡 Interpretación: Ingreso tiene patrón (MAR), Cantidad es aleatorio (MCAR)")

---

## 🔄 Ejercicio 2: Imputación y Comparación

**Objetivo**: Aplicar 3 estrategias y comparar resultados

In [0]:
# Estrategia 1: Imputación simple (media/mediana)
df_simple = ventas_missing.copy()

# Imputar numéricas con mediana
for col in df_simple.select_dtypes(include=[np.number]).columns:
    if df_simple[col].isnull().sum() > 0:
        df_simple[col].fillna(df_simple[col].median(), inplace=True)

print("✅ Estrategia 1: Imputación Simple con Mediana")
print(f"  Missing restantes: {df_simple.isnull().sum().sum()}")

In [0]:
# Estrategia 2: KNN Imputer
df_knn = ventas_missing.copy()
features_num = df_knn.select_dtypes(include=[np.number]).columns.tolist()

knn_imputer = KNNImputer(n_neighbors=5)
df_knn[features_num] = knn_imputer.fit_transform(df_knn[features_num])

print("✅ Estrategia 2: KNN Imputer (k=5)")
print(f"  Missing restantes: {df_knn.isnull().sum().sum()}")

In [0]:
# Comparar distribuciones antes/después
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original (sin missing para comparar)
ventas.head(1000)['ingreso'].plot(kind='hist', bins=30, alpha=0.7, ax=axes[0], color='blue')
axes[0].set_title("Original (sin missing)")
axes[0].set_xlabel("Ingreso")

# Simple
df_simple['ingreso'].plot(kind='hist', bins=30, alpha=0.7, ax=axes[1], color='orange')
axes[1].set_title("Imputación Simple (mediana)")
axes[1].set_xlabel("Ingreso")

# KNN
df_knn['ingreso'].plot(kind='hist', bins=30, alpha=0.7, ax=axes[2], color='green')
axes[2].set_title("KNN Imputer")
axes[2].set_xlabel("Ingreso")

plt.tight_layout()
plt.show()

print("\n💡 Observación: KNN preserva mejor la distribución original")

---

## 🔍 Ejercicio 3: Detección de Outliers

**Objetivo**: Comparar IQR, Z-score e Isolation Forest

In [0]:
# Método 1: IQR
df_clean = df_knn.copy()  # Usar datos imputados
columna = 'monto'

Q1 = df_clean[columna].quantile(0.25)
Q3 = df_clean[columna].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = df_clean[(df_clean[columna] < lower_bound) | (df_clean[columna] > upper_bound)]

print(f"✅ Método 1: IQR")
print(f"  Outliers detectados: {len(outliers_iqr)} ({len(outliers_iqr)/len(df_clean):.1%})")
print(f"  Rango normal: [{lower_bound:.2f}, {upper_bound:.2f}]")

In [0]:
# Método 2: Z-score
df_clean['z_score'] = stats.zscore(df_clean[columna])
outliers_zscore = df_clean[df_clean['z_score'].abs() > 3]

print(f"\n✅ Método 2: Z-Score")
print(f"  Outliers detectados: {len(outliers_zscore)} ({len(outliers_zscore)/len(df_clean):.1%})")

In [0]:
# Método 3: Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df_clean['anomaly'] = iso_forest.fit_predict(df_clean[[columna]])
outliers_iso = df_clean[df_clean['anomaly'] == -1]

print(f"\n✅ Método 3: Isolation Forest")
print(f"  Outliers detectados: {len(outliers_iso)} ({len(outliers_iso)/len(df_clean):.1%})")

print("\n📊 Comparación:")
print(f"  IQR: {len(outliers_iqr)} outliers")
print(f"  Z-Score: {len(outliers_zscore)} outliers")
print(f"  Isolation Forest: {len(outliers_iso)} outliers")

In [0]:
# Visualizar con boxplot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.boxplot(df_clean[columna])
plt.title(f"Boxplot de {columna}")
plt.ylabel("Monto")

plt.subplot(1, 2, 2)
plt.scatter(range(len(df_clean)), df_clean[columna], c=df_clean['anomaly'], cmap='coolwarm', alpha=0.5)
plt.axhline(upper_bound, color='red', linestyle='--', label='Límite IQR superior')
plt.axhline(lower_bound, color='red', linestyle='--', label='Límite IQR inferior')
plt.title("Outliers por Isolation Forest")
plt.xlabel("Index")
plt.ylabel("Monto")
plt.legend()

plt.tight_layout()
plt.show()

---

## ✅ Resumen

### 💡 Aprendizajes Clave

**Datos Faltantes:**
* ✅ MCAR vs MAR afectan estrategia de imputación
* ✅ KNN preserva mejor distribuciones que media/mediana
* ✅ MICE es más sofisticado pero más lento

**Outliers:**
* ✅ IQR es robusto y visual (boxplots)
* ✅ Z-score asume normalidad
* ✅ Isolation Forest es multivariado y flexible

**Decisiones:**
* ✅ Siempre investigar antes de eliminar
* ✅ Documentar transformaciones
* ✅ Validar impacto en modelos ML

---

### 🚀 Próximos Pasos

* Continúa con **Módulo 03: Análisis de Correlaciones**
* Aplica estos métodos en tus TPs
* Experimenta con datasets propios

---

**Universidad del Aconcagua**  
**Mendoza, Argentina 🇦🇷**